In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from tqdm import tqdm
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def extract_replicon_topology(genus_name, org_data_n):
    topo_data = []
    with tqdm(total = len(org_data_n['accession']), desc=f'{genus_name}({len(org_data_n['accession'])})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for i in range(len(org_data_n['accession'])):
            acc_n = org_data_n['accession'][i]
            pbar.update(1)
            handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
            seq_record = SeqIO.parse(handle, 'genbank')
            for record in seq_record:
                topo_data.append(pd.DataFrame([{'accession': f'{acc_n}-{record.id}', 'topology': record.annotations['topology']}]))
            handle.close()

    return pd.concat(topo_data, ignore_index=True)

In [3]:
for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    os.chdir(folder)
    replicon_data = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    topo_data = extract_replicon_topology(genus_name, org_data_n)

    replicon_data = pd.merge(replicon_data, topo_data, on='accession', how='left')
    
    os.chdir(folder)
    replicon_data.to_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv', index=False)

Escherichia(4204): 100%|████████████████████████████████████████| 4.20k/4.20k [39:02<00:00, 1.79B/s]
Klebsiella(3554): 100%|█████████████████████████████████████████| 3.55k/3.55k [36:32<00:00, 1.62B/s]
Staphylococcus(2423): 100%|█████████████████████████████████████| 2.42k/2.42k [10:42<00:00, 3.77B/s]
Pseudomonas(2343): 100%|████████████████████████████████████████| 2.34k/2.34k [24:40<00:00, 1.58B/s]
Bacillus(1976): 100%|███████████████████████████████████████████| 1.98k/1.98k [15:05<00:00, 2.18B/s]
Salmonella(1853): 100%|█████████████████████████████████████████| 1.85k/1.85k [15:04<00:00, 2.05B/s]
Streptococcus(1599): 100%|██████████████████████████████████████| 1.60k/1.60k [05:12<00:00, 5.12B/s]
Streptomyces(1359): 100%|███████████████████████████████████████| 1.36k/1.36k [18:13<00:00, 1.24B/s]
Acinetobacter(1234): 100%|██████████████████████████████████████| 1.23k/1.23k [07:29<00:00, 2.74B/s]
Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [01:00<00:00,